In [3]:
import numpy as np
from scipy.special import gamma

# ==========================================
# 1. ENVIRONMENT & PROBLEM SETUP
# ==========================================
START = np.array([0.0, 0.0])
TARGET = np.array([10.0, 10.0])
NUM_WAYPOINTS = 5

# Boundary limits representing the enclosing walls of the environment
X_MIN, X_MAX = 0.0, 10.0
Y_MIN, Y_MAX = 0.0, 10.0

# Define circular obstacles: [x_center, y_center, radius]
OBSTACLES = [
    np.array([3.0, 3.0, 1.5]),
    np.array([5.0, 6.0, 1.8]),
    np.array([7.0, 4.0, 1.2])
]

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def line_intersects_circle(p1, p2, circle):
    """Checks if a line segment between p1 and p2 intersects a circular obstacle."""
    cx, cy, r = circle
    c = np.array([cx, cy])

    d = p2 - p1
    f = p1 - c

    a = np.dot(d, d)
    # Safe check for consecutive identical waypoints to prevent division by zero
    if a == 0:
        return np.linalg.norm(p1 - c) <= r

    b = 2 * np.dot(f, d)
    cc = np.dot(f, f) - r**2

    discriminant = b**2 - 4*a*cc
    if discriminant < 0:
        return False

    discriminant = np.sqrt(discriminant)
    t1 = (-b - discriminant) / (2*a)
    t2 = (-b + discriminant) / (2*a)

    if (0 <= t1 <= 1) or (0 <= t2 <= 1):
        return True
    return False

def reconstruct_path(nest):
    waypoints = nest.reshape((NUM_WAYPOINTS, 2))
    return np.vstack([START, waypoints, TARGET])

def fitness_function(nest):
    path = reconstruct_path(nest)
    total_distance = 0.0
    penalty = 0.0

    for i in range(len(path) - 1):
        p1 = path[i]
        p2 = path[i+1]

        total_distance += np.linalg.norm(p2 - p1)

        for obs in OBSTACLES:
            if line_intersects_circle(p1, p2, obs):
                penalty += 1000.0

    return total_distance + penalty

# ==========================================
# 3. CUCKOO SEARCH ALGORITHM
# ==========================================
def levy_flight(d, beta=1.5):
    sigma = (gamma(1 + beta) * np.sin(np.pi * beta / 2) /
             (gamma((1 + beta) / 2) * beta * 2**((beta - 1) / 2)))**(1 / beta)
    u = np.random.normal(0, sigma, size=d)
    v = np.random.normal(0, 1, size=d)
    step = u / (np.abs(v)**(1 / beta))
    return step

def cuckoo_search(pop_size=25, max_iter=200, pa=0.25):
    dim = NUM_WAYPOINTS * 2

    nests = np.zeros((pop_size, dim))
    for i in range(pop_size):
        nests[i, 0::2] = np.random.uniform(X_MIN, X_MAX, NUM_WAYPOINTS)
        nests[i, 1::2] = np.random.uniform(Y_MIN, Y_MAX, NUM_WAYPOINTS)

    fitness = np.array([fitness_function(n) for n in nests])

    best_idx = np.argmin(fitness)
    best_nest = nests[best_idx].copy()
    best_fitness = fitness[best_idx]

    for t in range(max_iter):
        i = np.random.randint(0, pop_size)
        step = levy_flight(dim)
        step_size = 0.01 * step * (nests[i] - best_nest)

        new_cuckoo = nests[i] + np.random.normal(size=dim) * step_size
        new_cuckoo[0::2] = np.clip(new_cuckoo[0::2], X_MIN, X_MAX)
        new_cuckoo[1::2] = np.clip(new_cuckoo[1::2], Y_MIN, Y_MAX)

        new_fitness = fitness_function(new_cuckoo)

        j = np.random.randint(0, pop_size)
        if new_fitness < fitness[j]:
            nests[j] = new_cuckoo
            fitness[j] = new_fitness

        for k in range(pop_size):
            if np.random.rand() < pa:
                rand_idx1, rand_idx2 = np.random.choice(pop_size, 2, replace=False)
                step_size = np.random.rand() * (nests[rand_idx1] - nests[rand_idx2])
                nests[k] += step_size
                nests[k, 0::2] = np.clip(nests[k, 0::2], X_MIN, X_MAX)
                nests[k, 1::2] = np.clip(nests[k, 1::2], Y_MIN, Y_MAX)
                fitness[k] = fitness_function(nests[k])

        current_best_idx = np.argmin(fitness)
        if fitness[current_best_idx] < best_fitness:
            best_fitness = fitness[current_best_idx]
            best_nest = nests[current_best_idx].copy()

    return best_nest, best_fitness

# ==========================================
# 4. EXECUTION
# ==========================================
if __name__ == "__main__":
    best_solution, best_score = cuckoo_search(pop_size=40, max_iter=300)
    optimized_path = reconstruct_path(best_solution)

    print("--- ENVIRONMENT CONFIGURATION ---")
    print(f"Wall Boundaries: X limits [{X_MIN:.1f}, {X_MAX:.1f}], Y limits [{Y_MIN:.1f}, {Y_MAX:.1f}]")
    print("Obstacles Coordinates (Center X, Center Y, Radius):")
    for idx, obs in enumerate(OBSTACLES):
        print(f"  Obstacle {idx + 1}: Center=[{obs[0]:.2f}, {obs[1]:.2f}], Radius={obs[2]:.2f}")

    print("\n--- OPTIMIZATION RESULTS ---")
    print(f"Final Path Score/Fitness: {best_score:.4f}")
    print("\nOptimized Waypoint Sequence:")
    for idx, point in enumerate(optimized_path):
        if idx == 0:
            print(f"  Start:  [{point[0]:.2f}, {point[1]:.2f}]")
        elif idx == len(optimized_path) - 1:
            print(f"  Target: [{point[0]:.2f}, {point[1]:.2f}]")
        else:
            print(f"  WP {idx}:  [{point[0]:.2f}, {point[1]:.2f}]")


--- ENVIRONMENT CONFIGURATION ---
Wall Boundaries: X limits [0.0, 10.0], Y limits [0.0, 10.0]
Obstacles Coordinates (Center X, Center Y, Radius):
  Obstacle 1: Center=[3.00, 3.00], Radius=1.50
  Obstacle 2: Center=[5.00, 6.00], Radius=1.80
  Obstacle 3: Center=[7.00, 4.00], Radius=1.20

--- OPTIMIZATION RESULTS ---
Final Path Score/Fitness: 19.0637

Optimized Waypoint Sequence:
  Start:  [0.00, 0.00]
  WP 1:  [0.56, 0.29]
  WP 2:  [1.14, 0.02]
  WP 3:  [1.87, 1.38]
  WP 4:  [6.43, 0.00]
  WP 5:  [8.11, 0.42]
  Target: [10.00, 10.00]
